# We are comparing the baseline methods with our multi-agent method

## First, we load the questions that GPT-5-nano never got right in all 4 attempts

In [1]:
import os
from pathlib import Path
os.chdir(Path.cwd().parent)
from data_processing.data_analysis import select_problem_sample_for_model 
gpt5_nano="GPT-5-nano (high)"
df = select_problem_sample_for_model(gpt5_nano)
df

d:\conda\envs\nlp\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,Unnamed: 0,source,problem,competition,unique_problem_label,correct,parsed_answer,gold_answer,output_cost_per_tokens,problem_idx,cost,output_tokens,input_tokens,answer,user_message,idx_answer,model_config,model_name,input_cost_per_tokens,ten_percentile_group
5836,3928,NaN,Let $N$ denote the numbers of ordered triples ...,MathArena/aime_2025_outputs,MathArena/aime_2025: 15,False,147,735,0.4,15,0.024355,60875.0,107.0,\boxed{147},"Please reason step by step, and put your final...",0,openai/gpt-5-nano,GPT-5-nano (high),0.05,1
15668,3920,NaN,A plane $\mathcal{P}$ intersects a rectangular...,MathArena/hmmt_feb_2025_outputs,MathArena/hmmt_feb_2025: 30,False,sqrt(110),\sqrt{\frac{95}{24}},0.4,30,0.017099,42737.0,79.0,Let the six edges of the hexagonal cross-secti...,"Please reason step by step, and put your final...",0,openai/gpt-5-nano,GPT-5-nano (high),0.05,2
20160,2292,NaN,Consider all positive multiples of $77$ less t...,MathArena/cmimc_2025_outputs,MathArena/cmimc_2025: 5,False,25,194832,0.4,5,0.002802,6995.0,76.0,### Final answer\n\nReasoning:\n- The positive...,"Please reason step by step, and put your final...",0,openai/gpt-5-nano,GPT-5-nano (high),0.05,4
24100,2552,NaN,Let $A B C D E F$ be a convex cyclic hexagon. ...,MathArena/brumo_2025_outputs,MathArena/brumo_2025: 25,False,R,\frac{1+\sqrt{31}}{2},0.4,25,0.011091,27716.0,84.0,- Let the circle’s circumradius be R. For the ...,"Please reason step by step, and put your final...",0,openai/gpt-5-nano,GPT-5-nano (high),0.05,6
24116,2568,NaN,"$4$ bears - Aruno, Bruno, Cruno and Druno - ar...",MathArena/brumo_2025_outputs,MathArena/brumo_2025: 21,False,250,160,0.4,21,0.006556,16368.0,181.0,"Step 1: Let a, b, c, d be the four positive in...","Please reason step by step, and put your final...",0,openai/gpt-5-nano,GPT-5-nano (high),0.05,8


We have 5 problems that GPT-5 nano couldn't solve in various ranges of difficulty. We'll go from the easiest to hardest.

## Build a Statistics DataFrame

We build a dataframe comparing all the 3 methods to use this for a visualization later down the line.

In [2]:
import pandas as pd


UNIQUE_PROBLEM_LABEL = "unique_problem_label"
DIFFICULTY = "ten_percentile_group"
METHOD = "method"
TOTAL_TOKENS = "total_tokens"
CORRECT = "correct"
PARAMS = "params"
RESULT_DESC = "result_description"
cols = [UNIQUE_PROBLEM_LABEL, DIFFICULTY, METHOD, TOTAL_TOKENS, CORRECT, PARAMS, RESULT_DESC]

df_stats = pd.DataFrame(columns=cols)

## Problem 1 - Difficulty level 8

In [3]:
import pandas as pd
DIFFICULTY = "ten_percentile_group"
df_pruned = df[df[DIFFICULTY]==8].iloc[0][["unique_problem_label", "answer", "gold_answer", "ten_percentile_group", "problem"]]
df_pruned

unique_problem_label                             MathArena/brumo_2025: 21
answer                  Step 1: Let a, b, c, d be the four positive in...
gold_answer                                                           160
ten_percentile_group                                                    8
problem                 $4$ bears - Aruno, Bruno, Cruno and Druno - ar...
Name: 24116, dtype: object

In [ ]:
stat_dict = {
  UNIQUE_PROBLEM_LABEL: df_pruned[UNIQUE_PROBLEM_LABEL],
  DIFFICULTY: df_pruned[DIFFICULTY]
  }
stat_dict

REFLEXION = "reflexion"
TOT = "tot"
RS_BASIC = "rejection_sampling_basic"
RS_SUM = "rejection_sampling_summarized"

problem_rows = {
  REFLEXION: stat_dict.copy(),
  TOT: stat_dict.copy(),
  RS_BASIC: stat_dict.copy(),
  RS_SUM: stat_dict.copy()
  }

problem_rows

{'reflexion': {'unique_problem_label': 'MathArena/brumo_2025: 21',
  'ten_percentile_group': 8},
 'tot': {'unique_problem_label': 'MathArena/brumo_2025: 21',
  'ten_percentile_group': 8},
 'rejection_sampling_basic': {'unique_problem_label': 'MathArena/brumo_2025: 21',
  'ten_percentile_group': 8},
 'rejection_sampling_summarized': {'unique_problem_label': 'MathArena/brumo_2025: 21',
  'ten_percentile_group': 8}}

In [5]:
import textwrap

first_problem_description = df_pruned["problem"]

first_problem_gold_answer = df_pruned["gold_answer"]
print(textwrap.fill(text=first_problem_description, width=80))
print("answer", first_problem_gold_answer)

$4$ bears - Aruno, Bruno, Cruno and Druno - are each given a card with a
positive integer and are told that the sum of their $4$ numbers is $17$. They
cannot show each other their cards, but discuss a series of observations in the
following order:  Aruno: "I think it is possible that the other three bears all
have the same card." Bruno: "At first, I thought it was possible for the other
three bears to have the same card. Now I know it is impossible for them to have
the same card." Cruno: "I think it is still possible that the other three bears
have the same card." Druno: "I now know what card everyone has." What is the
product of their four card values?
answer 160


### Reflexion attempt

In [6]:
from multi_agent.multi_agent import Problem
from models.prompt_template import Reflexion_Solver, Reflector
problem = problem = Problem(
    roles=[Reflexion_Solver, Reflector],  # add Solver if you have one
    problem_descr=first_problem_description,
    answer=first_problem_gold_answer
)

print(problem)

Welcome Reflexion_Solver, and Reflector. Together, you should solve the
following problem: >> $4$ bears - Aruno, Bruno, Cruno and Druno - are each given
a card with a positive integer and are told that the sum of their $4$ numbers is
$17$. They cannot show each other their cards, but discuss a series of
observations in the following order:  Aruno: "I think it is possible that the
other three bears all have the same card." Bruno: "At first, I thought it was
possible for the other three bears to have the same card. Now I know it is
impossible for them to have the same card." Cruno: "I think it is still possible
that the other three bears have the same card." Druno: "I now know what card
everyone has." What is the product of their four card values?.<<  "When you are
done, you should submidt your answer as: ANSWER: <your answer>.  No latex
formatting, just the raw number/numbers or strings at the very end.  Before you
start sharing your toughts, give a little summary of the conversation so

In [6]:
# Getting the GPT-5-nano model

from models.azure_api import Client 
api_version = "2024-12-01-preview"
model_name="gpt-5-nano"


client = Client(
  api_version=api_version
)

model = client.select_model(
  model_name=model_name

)

In [ ]:
from baselines.reflexion.reflexion import ReflexionAgent, ReflexionStrategy, evaluator_fn
from functools import partial
eval = partial(evaluator_fn, model=model)

agent = ReflexionAgent(
    llm=model,
    strategy=ReflexionStrategy.LAST_ATTEMPT_AND_REFLEXION,
    evaluator_fn=eval,
    reflector_prompt=Reflector,
    max_attempts=3
)

solution = agent.solve(problem.problem_description)
print("\nFINAL SOLUTION:\n", solution)
print("Number of tokens used", model.num_tokens)


=== Attempt 1 ===
Let a, b, c, d be the four positive integers on the cards, with a + b + c + d = 17.

1) Aruno: “I think it is possible that the other three bears have the same card.”
- This means there exists x > 0 such that b = c = d = x and a + 3x = 17.
- Hence a = 17 − 3x > 0, so x can be 1,2,3,4,5, giving a ∈ {14,11,8,5,2}.
- Equivalently, Aruno’s a must be in A = {2, 5, 8, 11, 14}.

2) Bruno: “At first, I thought it was possible for the other three bears to have the same card. Now I know it is impossible for them to have the same card.”
- Initially (before Aruno spoke) Bruno would think a=c=d could happen iff there exists x > 0 with b + 3x = 17, i.e., 17 − b is divisible by 3.
  This requires b ∈ {2, 5, 8, 11, 14}.
- After hearing Aruno’s statement, Bruno must conclude that no a ∈ A can equal x = (17 − b)/3. So (17 − b)/3 ∉ A.
- Checking b ∈ {2,5,8,11,14}:
  - b = 2 → x = 5 ∈ A (possible)  
  - b = 5 → x = 4 ∉ A (impossible now)
  - b = 8 → x = 3 ∉ A (impossible now)
  - b = 11

The Reflexion agent solved this problem in 1 attempt

In [41]:
usages = [
    {"total_tokens": 14024},
    {"total_tokens": 5459},
    {"total_tokens": 23992},
    {"total_tokens": 8165},
]

total_tokens = sum(u["total_tokens"] for u in usages)

print("Total tokens:", total_tokens)


Total tokens: 51640


In [6]:
problem_rows[REFLEXION][TOTAL_TOKENS] = 51640
problem_rows[REFLEXION][CORRECT] = True
problem_rows[REFLEXION][PARAMS] = "max_attempt=3"

### Tree of thought attempt

In [ ]:
from baselines.tot.math_arena_tot_setup import ToTConfig, run_math_arena_tot
from openai import AzureOpenAI

config = ToTConfig(
    key_env_name="AZURE_OPENAI_API_KEY",
    endpoint_env_name= "AZURE_OPENAI_ENDPOINT",
    model_name=model_name,
    n_evaluate_sample=4,
    n_select_sample=4,
    n_generate_sample=4,
    steps=4,
    api_version=api_version,
    client_type=AzureOpenAI
  )

ys, infos = run_math_arena_tot(ToTConfig=config, problem_descr=first_problem_description, answer=first_problem_gold_answer)

print(ys)
print(infos)
from baselines.tot.tree_of_thought_llm_master.src.tot.models import gpt_usage
usage = gpt_usage()
print(usage)

path d:\NLP-group-15
>>>tries to call client.. Attempt> 0 <<<
>>>succesfully called client<<<
>>>tries to call client.. Attempt> 0 <<<
>>>succesfully called client<<<
-- new_ys --: ("- Step 1: From Aruno's statement, a is such that 17 − a can be written as 3t with t a positive integer. So 17 − a ≡ 0 (mod 3) and 17 − a > 0, giving a ∈ {2, 5, 8, 11, 14}.\n\n- Step 2: Bruno says: initially it was possible for A, C, D to be equal, but after hearing Aruno, it’s now impossible. If A, C, D were equal, then a = c = d = t and 17 − b = 3t. For such a triple to be consistent with Aruno’s constraint a ∈ {2, 5, 8, 11, 14}, t must be in {2, 5, 8, 11, 14}. But 17 − b must also be 3t, so 3t ≤ 16, giving t ∈ {2, 5} and hence b ∈ {11, 2}. Since Bruno now says it’s impossible, b ≠ 2, 11. Together with the initial requirement that 17 − b is divisible by 3 (for the initial possibility), we get b ∈ {5, 8, 14}.\n\n- Step 3: Cruno says it’s still possible that A, B, D are equal. If a = b = d = t, then 17 − c 

The tree of thought starts out with a wrong answer but eventually gets to the right answer after branching a few times.

In [7]:
problem_rows[TOT][TOTAL_TOKENS] = 148288 + 44895
problem_rows[TOT][CORRECT] = True
problem_rows[TOT][PARAMS] = "all=4"

### Solver-Rejector method - The proposed method

In [9]:
from collections import defaultdict

def compute_token_cost(model) -> defaultdict:
    token_counts = defaultdict(int)   

    for run_step in model.num_tokens:
        usage = run_step
        token_counts["total"] += usage.total_tokens

    return token_counts

In [17]:
from multi_agent.multi_agent import Role, Problem, conversation, rank_answer
from models.prompt_template import Solver, Rejector

from models.azure_api import Client
api_version = "2024-12-01-preview"
model_name="gpt-5-nano"

client = Client(
  api_version=api_version
)

model = client.select_model(
  model_name=model_name

)

first_problem = Problem(
  roles=[Solver, Rejector],
  problem_descr=first_problem_description, 
  answer=first_problem_gold_answer
  
)

messages, raw, path = conversation(model=model, name="GPT-5_nano_chat_Problem1" ,n_steps=10, problem=first_problem)
rank_answer(model=model, conversation=messages)
print("Number of tokens used", model.num_tokens)
print(compute_token_cost(model))


STEP 0: 

Role: Solver
 You solve problems.  You try to reason step by step. You are not too confident
in your answers (in the sense you are open to be wrong), but rather you rely on
fully fleshed out mathematical reasoning.  You try to explore many ideas.
Everytime you speak you will propose a fresh answer.  You dont submit the same
answer twice. Everytime you come with a new answer, you state all the previous
answers in a list in format of tuples: (Answer, short summary).  For example,  [
(780, induction on N, and lower bound on Z/N), (28/2, CLT of H and proof by
contradiction of Z>N) ] Then you check that your new proposal is not in that
list. If it is, you try again.  Use the early parts of your prompt as thinking
text, not "for science paper style" - meaning you can write your things and
doubts. Ex "I am thinking there could be a hint in the upper bound. I will check
it out. Ahh, I see I made a mistake. But now the size formula seems really
promising!"  "Then formalize and submit

Our method didn't rank the right answer as the highest in this case.

In [10]:
problem_rows[RS_BASIC][TOTAL_TOKENS] = 75289
problem_rows[RS_BASIC][CORRECT] = False
problem_rows[RS_BASIC][PARAMS] = "n_steps=10"

In [11]:
problem_rows

{'reflexion': {'unique_problem_label': 'MathArena/brumo_2025: 21',
  'ten_percentile_group': 8,
  'total_tokens': 51640,
  'correct': True,
  'params': 'max_attempt=3'},
 'tot': {'unique_problem_label': 'MathArena/brumo_2025: 21',
  'ten_percentile_group': 8,
  'total_tokens': 193183,
  'correct': True,
  'params': 'all=4'},
 'rejection_sampling_basic': {'unique_problem_label': 'MathArena/brumo_2025: 21',
  'ten_percentile_group': 8,
  'total_tokens': 75289,
  'correct': False,
  'params': 'n_steps=10'},
 'rejection_sampling_summarized': {'unique_problem_label': 'MathArena/brumo_2025: 21',
  'ten_percentile_group': 8}}

In [16]:
import summerized_conv
from summerized_conv import summarized_rejection_sampling_azure
from multi_agent.multi_agent import Role, Problem, rank_answer
from models.prompt_template import Solver, Rejector
from models.azure_api import Client
from importlib import reload
reload(summerized_conv)

client = Client(
  api_version=api_version
)

model = client.select_model(
  model_name=model_name

)

problem = Problem(
  roles=[Solver, Rejector],
  problem_descr=first_problem_description, 
  answer=first_problem_gold_answer
  
)

msg = summarized_rejection_sampling_azure(
  model=model,
  name=f"{model_name}_problem_1",
  n_steps=10,
  problem=problem
)

rank_answer(model=model, conversation=msg)
print(compute_token_cost(model))

Iteration 0

Solver: 
Here's a quick summary of the conversation so far:
- Four bears have positive integers A,B,C,D with A+B+C+D=17. They speak in order, with each statement constraining possible values based on what they could deduce given earlier statements.
- Analyzing the logical implications step by step (Aruno’s claim about the other three being equal, Bruno’s refined claim, Cruno’s still-possible triple-equal scenario, and finally Druno’s deduction) leads to a unique assignment for the four cards.
- The consistent solution is A=5, B=5, C=2, D=5, which sums to 17 and satisfies all statements.

Current suggested answers:
[]

New proposal:
- Reasoning outline (concise, not full chain-of-thought): 
  - Aruno’s possibility condition implies A ∈ {2,5,8,11,14}.
  - Bruno’s shift after Aruno’s remark implies B ∈ {5,8,14} (the cases B=2 or 11 would allow a triple-equal scenario given Aruno’s constraint).
  - Cruno states it’s still possible for A=B=D to be equal. For A=B=D=y, we must ha

In [12]:
problem_rows[RS_SUM][TOTAL_TOKENS] =  206756
problem_rows[RS_SUM][CORRECT] =  False
problem_rows[RS_SUM][PARAMS] =  "n_steps=10"
problem_rows[RS_SUM][RESULT_DESC] =  "2nd"

In [13]:
## Problem 
for method, row in problem_rows.items():
  df_row = pd.DataFrame([row])
  df_row[METHOD] = method
  df_stats = pd.concat([df_stats, df_row])
df_stats

,unique_problem_label,ten_percentile_group,method,total_tokens,correct,params,result_description
0,MathArena/brumo_2025: 21,8,reflexion,51640,True,max_attempt=3,NaN
0,MathArena/brumo_2025: 21,8,tot,193183,True,all=4,NaN
0,MathArena/brumo_2025: 21,8,rejection_sampling_basic,75289,False,n_steps=10,NaN
0,MathArena/brumo_2025: 21,8,rejection_sampling_summarized,206756,False,n_steps=10,2nd


## Problem 2 - Difficulty Level 6

In [14]:
import pandas as pd
DIFFICULTY = "ten_percentile_group"
df_pruned = df[df[DIFFICULTY]==6].iloc[0][["unique_problem_label", "answer", "gold_answer", "ten_percentile_group", "problem"]]
df_pruned

unique_problem_label                             MathArena/brumo_2025: 25
answer                  - Let the circle’s circumradius be R. For the ...
gold_answer                                         \frac{1+\sqrt{31}}{2}
ten_percentile_group                                                    6
problem                 Let $A B C D E F$ be a convex cyclic hexagon. ...
Name: 24100, dtype: object

In [15]:
import textwrap

problem_description = df_pruned["problem"]

problem_gold_answer = df_pruned["gold_answer"]
print(textwrap.fill(text=problem_description, width=80))
print("answer", problem_gold_answer)

Let $A B C D E F$ be a convex cyclic hexagon. Suppose that $A B=D E=\sqrt{5}, B
C=E F=3$, and $C D=F A=\sqrt{20}$. Compute the circumradius of $A B C D E F$.
answer \frac{1+\sqrt{31}}{2}


In [16]:
stat_dict = {
  UNIQUE_PROBLEM_LABEL: df_pruned[UNIQUE_PROBLEM_LABEL],
  DIFFICULTY: df_pruned[DIFFICULTY]
  }
stat_dict

problem_rows = {
  REFLEXION: stat_dict.copy(),
  TOT: stat_dict.copy(),
  RS_BASIC: stat_dict.copy(),
  RS_SUM: stat_dict.copy()
  }

problem_rows

{'reflexion': {'unique_problem_label': 'MathArena/brumo_2025: 25',
  'ten_percentile_group': 6},
 'tot': {'unique_problem_label': 'MathArena/brumo_2025: 25',
  'ten_percentile_group': 6},
 'rejection_sampling_basic': {'unique_problem_label': 'MathArena/brumo_2025: 25',
  'ten_percentile_group': 6},
 'rejection_sampling_summarized': {'unique_problem_label': 'MathArena/brumo_2025: 25',
  'ten_percentile_group': 6}}

### Reflexion Attempt

In [18]:
from multi_agent.multi_agent import Problem
from models.prompt_template import Reflexion_Solver, Reflector
problem = problem = Problem(
    roles=[Reflexion_Solver, Reflector],  # add Solver if you have one
    problem_descr=problem_description,
    answer=problem_gold_answer
)

print(problem)

Welcome Reflexion_Solver, and Reflector. Together, you should solve the
following problem: >> Let $A B C D E F$ be a convex cyclic hexagon. Suppose that
$A B=D E=\sqrt{5}, B C=E F=3$, and $C D=F A=\sqrt{20}$. Compute the circumradius
of $A B C D E F$..<<  "When you are done, you should submidt your answer as:
ANSWER: <your answer>.  No latex formatting, just the raw number/numbers or
strings at the very end.  Before you start sharing your toughts, give a little
summary of the conversation so far.  Give a list of the currently suggested
answers. Everytime you propose an aswer, check this list.  You proposal cannot
be in this this list. Try again and submit a new unique answer."


In [23]:
# Getting the GPT-5-nano model

from models.azure_api import Client 
api_version = "2024-12-01-preview"
model_name="gpt-5-nano"


client = Client(
  api_version=api_version
)

model = client.select_model(
  model_name=model_name

)

In [ ]:
from baselines.reflexion.reflexion import ReflexionAgent, ReflexionStrategy, evaluator_fn
from functools import partial
eval = partial(evaluator_fn, model=model)

agent = ReflexionAgent(
    llm=model,
    strategy=ReflexionStrategy.LAST_ATTEMPT_AND_REFLEXION,
    evaluator_fn=eval,
    reflector_prompt=Reflector,
    max_attempts=3
)

solution = agent.solve(problem.problem_description)
print("\nFINAL SOLUTION:\n", solution)
print("Number of tokens used", model.num_tokens)



=== Attempt 1 ===
Let the hexagon A B C D E F be cyclic with center O and circumradius R.

- The central angles subtending the sides satisfy α = ∠AOB, β = ∠BOC, γ = ∠COD, and since AB = DE, BC = EF, CD = FA, we have α = ∠AOB = ∠DOE, β = ∠BOC = ∠EOF, γ = ∠COD = ∠FOA. Hence the six central angles are α, β, γ, α, β, γ and α + β + γ = 2π/2 = π, i.e. α + β + γ = π.

- Let x = α/2, y = β/2, z = γ/2. Then x, y, z > 0 and x + y + z = π/2. The given side lengths give
  AB = 2R sin x = √5,
  BC = 2R sin y = 3,
  CD = 2R sin z = 2√5.

  Thus sin x = √5/(2R), sin y = 3/(2R), sin z = √5/R.

- Since x + y + z = π/2, we have sin(x + y) = cos z. Also
  sin(x + y) = sin x cos y + cos x sin y,
  with cos y = √(1 − sin^2 y) and cos x = √(1 − sin^2 x), cos z = √(1 − sin^2 z).

  Substituting sin x, sin y, sin z gives the equation
  (√5/(2R)) √(1 − 9/(4R^2)) + (3/(2R)) √(1 − 5/(4R^2)) = √(1 − 5/R^2).

- Let t = R^2. After clearing radicals and simplifying (a routine but lengthy algebraic elimination), one

This solved the problem in 1 attempt.

In [50]:
usages = [
    {"total_tokens": 22066},
    {"total_tokens": 4633},
]

total_tokens = sum(u["total_tokens"] for u in usages)

print("Total tokens:", total_tokens)


Total tokens: 26699


In [17]:
problem_rows[REFLEXION][TOTAL_TOKENS] = 26699
problem_rows[REFLEXION][CORRECT] = True
problem_rows[REFLEXION][PARAMS] = "max_attempt=3"

### Tree of thought attempt

In [17]:
from dotenv import load_dotenv
from baselines.tot.math_arena_tot_setup import ToTConfig, run_math_arena_tot
from openai import AzureOpenAI

load_dotenv()
config = ToTConfig(
    key_env_name="AZURE_OPENAI_API_KEY",
    endpoint_env_name= "AZURE_OPENAI_ENDPOINT",
    model_name=model_name,
    n_evaluate_sample=3,
    n_select_sample=3,
    n_generate_sample=3,
    steps=3,
    api_version=api_version,
    client_type=AzureOpenAI
  )

ys, infos = run_math_arena_tot(ToTConfig=config, problem_descr=problem_description, answer=problem_gold_answer)

print(ys)
print(infos)
from baselines.tot.tree_of_thought_llm_master.src.tot.models import gpt_usage
usage = gpt_usage()
print(usage)

>>>tries to call client.. Attempt> 0 <<<
>>>succesfully called client<<<
>>>tries to call client.. Attempt> 0 <<<
>>>succesfully called client<<<
-- new_ys --: ('Answer: R = sqrt((16 + √31)/2)\n\nReasoning (sketch of steps):\n- Let α, β, γ be the central angles for AB, BC, CD, with AB=DE, BC=EF, CD=FA, so α=δ, β=ε, γ=ζ and α+β+γ=π.\n- Set a=α/2, b=β/2, c=γ/2. Then a+b+c=π/2.\n- From chord lengths: sin a = AB/(2R) = √5/(2R), sin b = BC/(2R) = 3/(2R), sin c = CD/(2R) = 2√5/(2R) = √5/R.\n- Since a+b = π/2 − c, cos c = sin(a+b) = sin a cos b + cos a sin b.\n- Compute cos a = √(1 − sin^2 a) = √(4R^2 − 5)/(2R), cos b = √(4R^2 − 9)/(2R), cos c = √(R^2 − 5)/R.\n- This yields the equation √5√(4R^2−9) + 3√(4R^2−5) = 4R√(R^2−5).\n- Put t = R^2 and solve, after squaring twice, to get (t − 1)(4t^2 − 64t + 225) = 0.\n- The admissible root with R^2 > 5 is t = 8 + (√31)/2.\n- Therefore R^2 = 8 + (√31)/2 and R = sqrt(8 + (√31)/2) = sqrt((16 + √31)/2). (Numerical value ≈ 3.283.)', '- Let α, β, γ be the 

Model's final answer was sqrt((16 + √31)/2).

So, Tree of Thought got the right answer

In [ ]:
import numpy as np
(1+np.sqrt(31))/2

np.float64(3.2838821814150108)

In [20]:
np.sqrt((16+np.sqrt(31))/2)

np.float64(3.283882181415011)

In [18]:
problem_rows[TOT][TOTAL_TOKENS] = 122878+14449
problem_rows[TOT][CORRECT] = True
problem_rows[TOT][PARAMS] = "all=3"


### Solver-Rejector method - The proposed method

In [ ]:
from multi_agent.multi_agent import Role, Problem, conversation, rank_answer
from models.prompt_template import Solver, Rejector

from models.azure_api import Client
api_version = "2024-12-01-preview"
model_name="gpt-5-nano"

client = Client(
  api_version=api_version
)

model = client.select_model(
  model_name=model_name

)

problem = Problem(
  roles=[Solver, Rejector],
  problem_descr=problem_description, 
  answer=problem_gold_answer
  
)

messages, raw, path = conversation(model=model, name="GPT-5_nano_chat_Problem2" ,n_steps=10, problem=problem)
rank_answer(model=model, conversation=messages)
print("Number of tokens used", model.num_tokens)


STEP 0: 

Role: Solver
 You solve problems.  You try to reason step by step. You are not too confident
in your answers (in the sense you are open to be wrong), but rather you rely on
fully fleshed out mathematical reasoning.  You try to explore many ideas.
Everytime you speak you will propose a fresh answer.  You dont submit the same
answer twice. Everytime you come with a new answer, you state all the previous
answers in a list in format of tuples: (Answer, short summary).  For example,  [
(780, induction on N, and lower bound on Z/N), (28/2, CLT of H and proof by
contradiction of Z>N) ] Then you check that your new proposal is not in that
list. If it is, you try again.  Use the early parts of your prompt as thinking
text, not "for science paper style" - meaning you can write your things and
doubts. Ex "I am thinking there could be a hint in the upper bound. I will check
it out. Ahh, I see I made a mistake. But now the size formula seems really
promising!"  "Then formalize and submit

In this case, our method ranked the correct answer as the highest rank. Hence, our method worked best in this attempt among the two problems.

In [53]:
usages = [
    9394,
    2021,
    10773,
    2728,
    6955,
    2951,
    6312,
    3431,
    9095,
    3649,
    6876,
]

total_tokens = sum(usages)

print("Total tokens:", total_tokens)


Total tokens: 64185


In [19]:
problem_rows[RS_BASIC][TOTAL_TOKENS] = 64185
problem_rows[RS_BASIC][CORRECT] = True
problem_rows[RS_BASIC][PARAMS] = "n_steps=10"

In [30]:
import summerized_conv
from summerized_conv import summarized_rejection_sampling_azure
from multi_agent.multi_agent import Role, Problem, rank_answer
from models.prompt_template import Solver, Rejector
from models.azure_api import Client
from importlib import reload
reload(summerized_conv)

client = Client(
  api_version=api_version
)

model = client.select_model(
  model_name=model_name

)

problem = Problem(
  roles=[Solver, Rejector],
  problem_descr=problem_description, 
  answer=problem_gold_answer
  
)

msg = summarized_rejection_sampling_azure(
  model=model,
  name=f"{model_name}_problem_1",
  n_steps=10,
  problem=problem
)

rank_answer(model=model, conversation=msg)
print(compute_token_cost(model))

Iteration 0

Solver: 
Summary of the conversation so far:
- We’re given a convex cyclic hexagon ABCDEF with AB = DE = sqrt(5), BC = EF = 3, CD = FA = 2*sqrt(5). If we denote the central angles subtending AB, BC, CD by α, β, γ, then the opposite sides correspond to the same angles: α, β, γ, α, β, γ, so α + β + γ = π.
- Let x = α/2, y = β/2, z = γ/2. Then sin x = AB/(2R) = sqrt(5)/(2R), sin y = BC/(2R) = 3/(2R), sin z = CD/(2R) = sqrt(5)/R = 2 sin x.
- Since x + y + z = π/2 and sin z = cos(x + y), we get a relation cos x cos y − sin x sin y = 2 sin x.
- With a = sin x and b = sin y, this reduces to a nice algebraic condition which leads to 2R^3 − 17R − 15 = 0.
- The circumradius R is the positive real root of this cubic, approximately R ≈ 3.2841.

Current suggested answers:
[
(3.2841, positive real root of 2R^3 − 17R − 15 = 0)
]

Proposed new answer (not in the list yet):
R is the positive real root of 2R^3 − 17R − 15 = 0, which gives R ≈ 3.2841.

ANSWER: 3.2841

Rejector: 
Rejector: You

In [20]:
problem_rows[RS_SUM][TOTAL_TOKENS] =  201577
problem_rows[RS_SUM][CORRECT] =  True
problem_rows[RS_SUM][PARAMS] =  "n_steps=10"

## Token cost

In [28]:
for method, row in problem_rows.items():
  df_row = pd.DataFrame([row])
  df_row[METHOD] = method
  df_stats = pd.concat([df_stats, df_row])
df_stats

,unique_problem_label,ten_percentile_group,method,total_tokens,correct,params,result_description
0,MathArena/brumo_2025: 21,8,reflexion,51640,True,max_attempt=3,NaN
0,MathArena/brumo_2025: 21,8,tot,193183,True,all=4,NaN
0,MathArena/brumo_2025: 21,8,rejection_sampling_basic,75289,False,n_steps=10,NaN
0,MathArena/brumo_2025: 21,8,rejection_sampling_summarized,206756,False,n_steps=10,2nd
0,MathArena/brumo_2025: 25,6,reflexion,26699,True,max_attempt=3,NaN
0,MathArena/brumo_2025: 25,6,tot,137327,True,all=3,NaN
0,MathArena/brumo_2025: 25,6,rejection_sampling_basic,64185,True,n_steps=10,NaN
0,MathArena/brumo_2025: 25,6,rejection_sampling_summarized,201577,True,n_steps=10,NaN


In [27]:
df_stats = df_stats.iloc[:-4]
df_stats

,unique_problem_label,ten_percentile_group,method,total_tokens,correct,params,result_description
0,MathArena/brumo_2025: 21,8,reflexion,51640,True,max_attempt=3,NaN
0,MathArena/brumo_2025: 21,8,tot,193183,True,all=4,NaN
0,MathArena/brumo_2025: 21,8,rejection_sampling_basic,75289,False,n_steps=10,NaN
0,MathArena/brumo_2025: 21,8,rejection_sampling_summarized,206756,False,n_steps=10,2nd


## Problem 3 - Difficulty Level 4

In [29]:
import pandas as pd
DIFFICULTY = "ten_percentile_group"
df_pruned = df[df[DIFFICULTY]==4].iloc[0][["unique_problem_label", "answer", "gold_answer", "ten_percentile_group", "problem"]]
df_pruned

unique_problem_label                              MathArena/cmimc_2025: 5
answer                  ### Final answer\n\nReasoning:\n- The positive...
gold_answer                                                        194832
ten_percentile_group                                                    4
problem                 Consider all positive multiples of $77$ less t...
Name: 20160, dtype: object

In [30]:
stat_dict = {
  UNIQUE_PROBLEM_LABEL: df_pruned[UNIQUE_PROBLEM_LABEL],
  DIFFICULTY: df_pruned[DIFFICULTY]
  }
stat_dict

problem_rows = {
  REFLEXION: stat_dict.copy(),
  TOT: stat_dict.copy(),
  RS_BASIC: stat_dict.copy(),
  RS_SUM: stat_dict.copy()
  }

problem_rows

{'reflexion': {'unique_problem_label': 'MathArena/cmimc_2025: 5',
  'ten_percentile_group': 4},
 'tot': {'unique_problem_label': 'MathArena/cmimc_2025: 5',
  'ten_percentile_group': 4},
 'rejection_sampling_basic': {'unique_problem_label': 'MathArena/cmimc_2025: 5',
  'ten_percentile_group': 4},
 'rejection_sampling_summarized': {'unique_problem_label': 'MathArena/cmimc_2025: 5',
  'ten_percentile_group': 4}}

In [31]:
import textwrap

problem_description = df_pruned["problem"]

problem_gold_answer = df_pruned["gold_answer"]
print(textwrap.fill(text=problem_description, width=80))
print("answer", problem_gold_answer)

Consider all positive multiples of $77$ less than $1,000,000$. What is the sum
of all the odd digits that show up?
answer 194832


### Reflexion Attempt

In [ ]:
from multi_agent.multi_agent import Problem
from models.prompt_template import Reflexion_Solver, Reflector
problem = problem = Problem(
    roles=[Reflexion_Solver, Reflector],  # add Solver if you have one
    problem_descr=problem_description,
    answer=problem_gold_answer
)

print(problem)

Welcome Reflexion_Solver, and Reflector. Together, you should solve the
following problem: >> Consider all positive multiples of $77$ less than
$1,000,000$. What is the sum of all the odd digits that show up?.<<  "When you
are done, you should submidt your answer as: ANSWER: <your answer>.  No latex
formatting, just the raw number/numbers or strings at the very end.  Before you
start sharing your toughts, give a little summary of the conversation so far.
Give a list of the currently suggested answers. Everytime you propose an aswer,
check this list.  You proposal cannot be in this this list. Try again and submit
a new unique answer."


In [37]:
# Getting the GPT-5-nano model

from models.azure_api import Client 
api_version = "2024-12-01-preview"
model_name="gpt-5-nano"


client = Client(
  api_version=api_version
)

model = client.select_model(
  model_name=model_name

)

In [ ]:
from baselines.reflexion.reflexion import ReflexionAgent, ReflexionStrategy, evaluator_fn
from functools import partial
eval = partial(evaluator_fn, model=model)

agent = ReflexionAgent(
    llm=model,
    strategy=ReflexionStrategy.LAST_ATTEMPT_AND_REFLEXION,
    evaluator_fn=eval,
    reflector_prompt=Reflector,
    max_attempts=3
)

solution = agent.solve(problem.problem_description)
print("\nFINAL SOLUTION:\n", solution)
print("Number of tokens used", model.num_tokens)


=== Attempt 1 ===
Step-by-step solution:

1) Let the multiples be m_k = 77k for k = 1, 2, ..., 12,987 (since 77 × 12,987 = 999,999).

2) Look at the last digit of m_k. Since 77 ≡ 7 (mod 10), the last digit of m_k is the same as the last digit of 7k, i.e., m_k ≡ 7k (mod 10).

3) Because gcd(7, 10) = 1, the sequence 7k mod 10 (as k runs through any 10 consecutive integers) takes every residue 0 through 9 exactly once. In particular, among the first 10 multiples, the last digits are a permutation of 0,1,2,3,4,5,6,7,8,9.

4) Therefore, among all multiples m_k (k = 1 to 12,987), every digit 0–9 appears at least once as the last digit. In particular, the odd digits 1, 3, 5, 7, and 9 all appear somewhere in those numbers.

5) The problem asks for the sum of all the odd digits that show up (i.e., the distinct odd digits that appear). Those digits are {1, 3, 5, 7, 9}.

6) Sum = 1 + 3 + 5 + 7 + 9 = 25.

Answer: 25
Score: true
Feedback: Verdict: Correct.

Feedback: The solution correctly notes t

Here, in this case, Reflexion couldn't get to the correct answer!

In [32]:
problem_rows[REFLEXION][TOTAL_TOKENS] = 3006 + 3133
problem_rows[REFLEXION][CORRECT] = False
problem_rows[REFLEXION][PARAMS] = "max_attempt=3"

### Tree of thought attempt

In [25]:
from dotenv import load_dotenv
from baselines.tot.math_arena_tot_setup import ToTConfig, run_math_arena_tot
from openai import AzureOpenAI

load_dotenv()
config = ToTConfig(
    key_env_name="AZURE_OPENAI_API_KEY",
    endpoint_env_name= "AZURE_OPENAI_ENDPOINT",
    model_name=model_name,
    n_evaluate_sample=3,
    n_select_sample=3,
    n_generate_sample=3,
    steps=3,
    api_version=api_version,
    client_type=AzureOpenAI
  )

ys, infos = run_math_arena_tot(ToTConfig=config, problem_descr=problem_description, answer=problem_gold_answer)

print(ys)
print(infos)
from baselines.tot.tree_of_thought_llm_master.src.tot.models import gpt_usage
usage = gpt_usage()
print(usage)

path /Users/nishadjahan/Documents/Scholarship/Denmark/Study/NLP/project/NLP-group-15
>>>tries to call client.. Attempt> 0 <<<
>>>succesfully called client<<<
>>>tries to call client.. Attempt> 0 <<<
>>>succesfully called client<<<
-- new_ys --: ('Step: The multiples are 77n with 1 ≤ n ≤ floor(999999/77) = 12987, since 12987·77 = 999,999 < 1,000,000.\n\nNext steps:\n- The number 999,999 appears (n = 12,987), so the digit 9 appears.\n- The number 77 appears (n = 1), so the digit 7 appears.\n- The number 154 appears (n = 2), so digits 1 and 5 appear.\n- The number 231 appears (n = 3), so digit 3 appears.\n\nThus all odd digits {1, 3, 5, 7, 9} occur among these multiples.\n\nSum of all odd digits that show up = 1 + 3 + 5 + 7 + 9 = 25.\n\nAnswer: 25', 'Answer: 25', 'Answer: 25')
-- sol values --: (3, 0, 0)
-- choices --: ['Step: The multiples are 77n with 1 ≤ n ≤ floor(999999/77) = 12987, since 12987·77 = 999,999 < 1,000,000.\n\nNext steps:\n- The number 999,999 appears (n = 12,987), so the

In this case, Tree of Thought did not get the right answer.

In [33]:
problem_rows[TOT][TOTAL_TOKENS] = 86929+5471
problem_rows[TOT][CORRECT] = False
problem_rows[TOT][PARAMS] = "all=3"

### Solver-Rejector method - The proposed method

In [ ]:
from multi_agent.multi_agent import Role, Problem, conversation, rank_answer
from models.prompt_template import Solver, Rejector

from models.azure_api import Client
api_version = "2024-12-01-preview"
model_name="gpt-5-nano"

client = Client(
  api_version=api_version
)

model = client.select_model(
  model_name=model_name

)

problem = Problem(
  roles=[Solver, Rejector],
  problem_descr=problem_description, 
  answer=problem_gold_answer
  
)

messages, raw, path = conversation(model=model, name="GPT-5_nano_chat_Problem2" ,n_steps=10, problem=problem)
rank_answer(model=model, conversation=messages)
print("Number of tokens used", model.num_tokens)


STEP 0: 

Role: Solver
 You solve problems.  You try to reason step by step. You are not too confident
in your answers (in the sense you are open to be wrong), but rather you rely on
fully fleshed out mathematical reasoning.  You try to explore many ideas.
Everytime you speak you will propose a fresh answer.  You dont submit the same
answer twice. Everytime you come with a new answer, you state all the previous
answers in a list in format of tuples: (Answer, short summary).  For example,  [
(780, induction on N, and lower bound on Z/N), (28/2, CLT of H and proof by
contradiction of Z>N) ] Then you check that your new proposal is not in that
list. If it is, you try again.  Use the early parts of your prompt as thinking
text, not "for science paper style" - meaning you can write your things and
doubts. Ex "I am thinking there could be a hint in the upper bound. I will check
it out. Ahh, I see I made a mistake. But now the size formula seems really
promising!"  "Then formalize and submit

Our method got the answer in the second attempt but ranked it 2nd. Only our method ranked close to the right answer 194832 while the other methods didn't.

In [34]:
totals = [
    3574, 5667, 19949, 3070, 8505,
    3525, 21227, 3403, 30227,
    3971, 15717
]

print(sum(totals))


118835


In [ ]:
problem_rows[RS_BASIC][TOTAL_TOKENS] = 118835
problem_rows[RS_BASIC][PARAMS] = "n_steps=10"
problem_rows[RS_BASIC][CORRECT] =  False
problem_rows[RS_BASIC][RESULT_DESC] = "2nd"

In [44]:
import summerized_conv
from summerized_conv import summarized_rejection_sampling_azure
from multi_agent.multi_agent import Role, Problem, rank_answer
from models.prompt_template import Solver, Rejector
from models.azure_api import Client
from importlib import reload
reload(summerized_conv)

client = Client(
  api_version=api_version
)

model = client.select_model(
  model_name=model_name

)

problem = Problem(
  roles=[Solver, Rejector],
  problem_descr=problem_description, 
  answer=problem_gold_answer
  
)

msg = summarized_rejection_sampling_azure(
  model=model,
  name=f"{model_name}_problem_1",
  n_steps=10,
  problem=problem
)

rank_answer(model=model, conversation=msg)

Iteration 0

Solver: 
Summary so far:
- We’re solving: take all positive multiples of 77 less than 1,000,000, and sum the odd digits that appear.
- There’s ambiguity whether to sum distinct odd digits or count every occurrence. The likely intended interpretation for a neat answer is the sum of distinct odd digits that appear.

Currently suggested answers:
[
]

New unique answer proposal:
(25, "sum of distinct odd digits that appear (1+3+5+7+9)")

Updated list of suggested answers:
[
(25, "sum of distinct odd digits that appear (1+3+5+7+9)")
]

Answer:
ANSWER: 25

Rejector: 
Rejector: Your answer is rejected. Your reasoning is wrong. Keep trying and stay explorative. Remember to list all proposed answers so far and not repeat any. Current proposed answers: [25]. Do not repeat. Use reasoning and accumulated knowledge, not guesses or bias from previous outcomes. Now propose a new unique answer.

Summarizer: 
Solver: Summary so far
- Problem: Sum all odd digits that appear among all positi

('Summary of the conversation so far:\n- Task: sum the odd digits that appear in decimal representations of all positive multiples of 77 below 1,000,000 (counting digits with multiplicity, unless stated otherwise by interpretation).\n- Several proposed numeric answers were given earlier, but none were accepted as final correct by the Rejector. The numbers include 25, 50, 123456, 32472, 194832, 227304, 360000, 194805, 194844, 195060.\n- You now want a ranked list of these existing answers judged by plausibility, with brief rationale for each, not a new answer.\n\nRanking of existing answers by plausibility (from most plausible to least plausible), with brief justification:\n\n1) 25\n- Why it could be true: If one interprets “sum of all the odd digits that show up” as summing distinct odd digits that appear somewhere among all multiples (i.e., the set of odd digits that occur). Since the units digit of multiples of 77 cycles through all digits, all odd digits {1,3,5,7,9} appear at least 

In [48]:
print(compute_token_cost(model))

defaultdict(<class 'int'>, {'total': 207031})


In [36]:
problem_rows[RS_SUM][TOTAL_TOKENS] = 207031
problem_rows[RS_SUM][CORRECT] =  False
problem_rows[RS_SUM][PARAMS] =  "n_steps=10"
problem_rows[RS_SUM][RESULT_DESC] =  "3rd"

In [37]:
## Problem 
for method, row in problem_rows.items():
  df_row = pd.DataFrame([row])
  df_row[METHOD] = method
  df_stats = pd.concat([df_stats, df_row])
df_stats

,unique_problem_label,ten_percentile_group,method,total_tokens,correct,params,result_description
0,MathArena/brumo_2025: 21,8,reflexion,51640,True,max_attempt=3,NaN
0,MathArena/brumo_2025: 21,8,tot,193183,True,all=4,NaN
0,MathArena/brumo_2025: 21,8,rejection_sampling_basic,75289,False,n_steps=10,NaN
0,MathArena/brumo_2025: 21,8,rejection_sampling_summarized,206756,False,n_steps=10,2nd
0,MathArena/brumo_2025: 25,6,reflexion,26699,True,max_attempt=3,NaN
0,MathArena/brumo_2025: 25,6,tot,137327,True,all=3,NaN
0,MathArena/brumo_2025: 25,6,rejection_sampling_basic,64185,True,n_steps=10,NaN
0,MathArena/brumo_2025: 25,6,rejection_sampling_summarized,201577,True,n_steps=10,NaN
0,MathArena/cmimc_2025: 5,4,reflexion,6139,False,max_attempt=3,NaN
0,MathArena/cmimc_2025: 5,4,tot,92400,False,all=3,NaN


In [60]:
df_stats.iloc[10, 4] = False

## Problem 4 - Difficulty Level 2

In [38]:
import pandas as pd
DIFFICULTY = "ten_percentile_group"
df_pruned = df[df[DIFFICULTY]==2].iloc[0][["unique_problem_label", "answer", "gold_answer", "ten_percentile_group", "problem"]]
df_pruned

unique_problem_label                          MathArena/hmmt_feb_2025: 30
answer                  Let the six edges of the hexagonal cross-secti...
gold_answer                                          \sqrt{\frac{95}{24}}
ten_percentile_group                                                    2
problem                 A plane $\mathcal{P}$ intersects a rectangular...
Name: 15668, dtype: object

In [39]:
stat_dict = {
  UNIQUE_PROBLEM_LABEL: df_pruned[UNIQUE_PROBLEM_LABEL],
  DIFFICULTY: df_pruned[DIFFICULTY]
  }
stat_dict

problem_rows = {
  REFLEXION: stat_dict.copy(),
  TOT: stat_dict.copy(),
  RS_BASIC: stat_dict.copy(),
  RS_SUM: stat_dict.copy()
  }

problem_rows

{'reflexion': {'unique_problem_label': 'MathArena/hmmt_feb_2025: 30',
  'ten_percentile_group': 2},
 'tot': {'unique_problem_label': 'MathArena/hmmt_feb_2025: 30',
  'ten_percentile_group': 2},
 'rejection_sampling_basic': {'unique_problem_label': 'MathArena/hmmt_feb_2025: 30',
  'ten_percentile_group': 2},
 'rejection_sampling_summarized': {'unique_problem_label': 'MathArena/hmmt_feb_2025: 30',
  'ten_percentile_group': 2}}

In [40]:
import textwrap

problem_description = df_pruned["problem"]

problem_gold_answer = df_pruned["gold_answer"]
print(textwrap.fill(text=problem_description, width=80))
print("answer", problem_gold_answer)

A plane $\mathcal{P}$ intersects a rectangular prism at a hexagon which has side
lengths $45,66,63,55,54$, and 77, in that order. Compute the distance from the
center of the rectangular prism to $\mathcal{P}$.
answer \sqrt{\frac{95}{24}}


### Reflexion Attempt

In [ ]:
from multi_agent.multi_agent import Problem
from models.prompt_template import Reflexion_Solver, Reflector
problem = problem = Problem(
    roles=[Reflexion_Solver, Reflector],  # add Solver if you have one
    problem_descr=problem_description,
    answer=problem_gold_answer
)

print(problem)

Welcome Reflexion_Solver, and Reflector. Together, you should solve the
following problem: >> A plane $\mathcal{P}$ intersects a rectangular prism at a
hexagon which has side lengths $45,66,63,55,54$, and 77, in that order. Compute
the distance from the center of the rectangular prism to $\mathcal{P}$..<<
"When you are done, you should submidt your answer as: ANSWER: <your answer>.
No latex formatting, just the raw number/numbers or strings at the very end.
Before you start sharing your toughts, give a little summary of the conversation
so far.  Give a list of the currently suggested answers. Everytime you propose
an aswer, check this list.  You proposal cannot be in this this list. Try again
and submit a new unique answer."


In [57]:
# Getting the GPT-5-nano model

from models.azure_api import Client 
api_version = "2024-12-01-preview"
model_name="gpt-5-nano"


client = Client(
  api_version=api_version
)

model = client.select_model(
  model_name=model_name

)

In [ ]:
from baselines.reflexion.reflexion import ReflexionAgent, ReflexionStrategy, evaluator_fn
from functools import partial
eval = partial(evaluator_fn, model=model)

agent = ReflexionAgent(
    llm=model,
    strategy=ReflexionStrategy.LAST_ATTEMPT_AND_REFLEXION,
    evaluator_fn=eval,
    reflector_prompt=Reflector,
    max_attempts=100
)

solution = agent.solve(problem.problem_description)
print("\nFINAL SOLUTION:\n", solution)
print("Number of tokens used", model.num_tokens)


=== Attempt 1 ===
Let the rectangular prism be axis-aligned with side lengths 2X, 2Y, 2Z along the x-, y-, z-axes, respectively. A plane with equation ax + by + cz = d cuts the prism, producing a hexagonal cross-section. Each side of the hexagon lies on one of the six faces of the prism. The six sides, in order around the hexagon, correspond to the intersections with the faces x = ±X, y = ±Y, z = ±Z.

A formula for the length of the intersection of a plane with a given face is obtained by looking at the line of intersection with that face and restricting to the appropriate rectangle (the face). If we write the plane as ax + by + cz = d, and focus on the face x = X, the intersection in the (y,z) plane is given by by + cz = d − aX. The segment lies inside the rectangle |y| ≤ Y, |z| ≤ Z. The length of that segment is, up to which of the two possible regimes applies, one of two expressions:

- If the line cuts the lines y = ±Y, the length is L_x(+) = (Y/|c|)√(b^2 + c^2).
- If the line cut

Reflexion is unable to answer this question.

In [41]:
usages = [
    26732,
    5125,
    5768,
    22644,
    4544,
    7158,
    14261,
    7655,
    8255,
    41531,
    11246,
]

total_tokens = sum(usages)
print("Total tokens:", total_tokens)

Total tokens: 154919


In [42]:
problem_rows[REFLEXION][TOTAL_TOKENS] = 154919
problem_rows[REFLEXION][CORRECT] = False
problem_rows[REFLEXION][PARAMS] = "max_attempt=100"

### Tree of thought attempt

In [32]:
from dotenv import load_dotenv
from baselines.tot.math_arena_tot_setup import ToTConfig, run_math_arena_tot
from openai import AzureOpenAI

load_dotenv()
config = ToTConfig(
    key_env_name="AZURE_OPENAI_API_KEY",
    endpoint_env_name= "AZURE_OPENAI_ENDPOINT",
    model_name=model_name,
    n_evaluate_sample=3,
    n_select_sample=3,
    n_generate_sample=3,
    steps=3,
    api_version=api_version,
    client_type=AzureOpenAI
  )

ys, infos = run_math_arena_tot(ToTConfig=config, problem_descr=problem_description, answer=problem_gold_answer)

print(ys)
print(infos)
from baselines.tot.tree_of_thought_llm_master.src.tot.models import gpt_usage
usage = gpt_usage()
print(usage)

>>>tries to call client.. Attempt> 0 <<<
>>>succesfully called client<<<
>>>tries to call client.. Attempt> 0 <<<
>>>succesfully called client<<<
-- new_ys --: ('Answer: \\sqrt{110}\n\nReasoning (sketch):\n- Let the prism be [-a,a]×[-b,b]×[-c,c], and the plane P have unit normal n = (n_x, n_y, n_z) and distance d from the center, so its equation is n_x x + n_y y + n_z z = d.\n- The hexagonal cross-section intersects the six faces; denote the lengths on opposite faces along x by s1 and s4, along y by s2 and s5, and along z by s3 and s6. The problem gives the six side lengths in order, so the three differences between opposite sides are\n  Δ1 = |s4 − s1|, Δ2 = |s5 − s2|, Δ3 = |s6 − s3|.\n- A standard property of such a cross-section is that the difference between the lengths on opposite faces is proportional to the plane’s offset from the center along the respective axis: Δ1 = 2 d |n_x|, Δ2 = 2 d |n_y|, Δ3 = 2 d |n_z|.\n- Squaring and summing gives Δ1^2 + Δ2^2 + Δ3^2 = 4 d^2 (n_x^2 + n_y

The Tree of Thought method proposed the answer sqrt(110). But the right answer is sqrt(95/24). So, it didn't give the right answer.

In [43]:
problem_rows[TOT][TOTAL_TOKENS] = 206025+17901
problem_rows[TOT][CORRECT] = False
problem_rows[TOT][PARAMS] = "all=3"

### Solver-Rejector method - The proposed method

In [ ]:
from multi_agent.multi_agent import Role, Problem, conversation, rank_answer
from models.prompt_template import Solver, Rejector

from models.azure_api import Client
api_version = "2024-12-01-preview"
model_name="gpt-5-nano"

client = Client(
  api_version=api_version
)

model = client.select_model(
  model_name=model_name

)

problem = Problem(
  roles=[Solver, Rejector],
  problem_descr=problem_description, 
  answer=problem_gold_answer
  
)

messages, raw, path = conversation(model=model, name="GPT-5_nano_chat_Problem2" ,n_steps=100, problem=problem)
rank_answer(model=model, conversation=messages)
print("Number of tokens used", model.num_tokens)


STEP 0: 

Role: Solver
 You solve problems.  You try to reason step by step. You are not too confident
in your answers (in the sense you are open to be wrong), but rather you rely on
fully fleshed out mathematical reasoning.  You try to explore many ideas.
Everytime you speak you will propose a fresh answer.  You dont submit the same
answer twice. Everytime you come with a new answer, you state all the previous
answers in a list in format of tuples: (Answer, short summary).  For example,  [
(780, induction on N, and lower bound on Z/N), (28/2, CLT of H and proof by
contradiction of Z>N) ] Then you check that your new proposal is not in that
list. If it is, you try again.  Use the early parts of your prompt as thinking
text, not "for science paper style" - meaning you can write your things and
doubts. Ex "I am thinking there could be a hint in the upper bound. I will check
it out. Ahh, I see I made a mistake. But now the size formula seems really
promising!"  "Then formalize and submit

The Rejection sampling method didn't get the answer :(

In [71]:
totals = [
    10406, 4551, 8735, 3665, 9075, 2740, 9619, 3900, 6893, 3775,
    10783, 4405, 11057, 4789, 12654, 5156, 10214, 6637, 11602, 8938,
    9417, 8245, 9372, 7920, 10155, 8600, 9718, 10173, 11861, 10412,
    11762, 10787, 13953, 11233, 13743, 11804, 13226, 12095, 14485,
    13162, 15472, 14038, 16124, 14139, 15478, 15058, 15976, 15923,
    16863, 16336, 17818, 16529, 18585, 17900, 18963, 17841, 19051,
    18364, 19436, 18782, 20001, 18945, 19540, 20262, 20966, 19975,
    21425, 21200, 24562, 21099, 21697, 21314, 22134, 21437, 23353,
    22810, 25072, 22754, 24251, 23166, 24423, 23578, 25448, 24105,
    26961, 24609, 25810, 25574, 25910, 25348, 27076, 25637, 26586,
    26448, 27890, 27030, 28343, 27331, 28517, 27808, 29440
]

print(sum(totals))


1700228


In [44]:
problem_rows[RS_BASIC][TOTAL_TOKENS] = 1700228
problem_rows[RS_BASIC][CORRECT] = False
problem_rows[RS_BASIC][PARAMS] = "n_steps=100"
problem_rows[RS_BASIC][RESULT_DESC] = None

In [63]:
import summerized_conv
from summerized_conv import summarized_rejection_sampling_azure
from multi_agent.multi_agent import Role, Problem, rank_answer
from models.prompt_template import Solver, Rejector
from models.azure_api import Client
from importlib import reload
reload(summerized_conv)

client = Client(
  api_version=api_version
)

model = client.select_model(
  model_name=model_name

)

problem = Problem(
  roles=[Solver, Rejector],
  problem_descr=problem_description, 
  answer=problem_gold_answer
  
)

msg = summarized_rejection_sampling_azure(
  model=model,
  name=f"{model_name}_problem_4",
  n_steps=100,
  problem=problem
)

rank_answer(model=model, conversation=msg)
print(compute_token_cost(model))

Iteration 0

Solver: 
Summary of conversation so far:
- You asked me to act as Solver to solve a geometry problem: a plane cuts a rectangular prism forming a hexagon with side lengths 45, 66, 63, 55, 54, 77 in that order. Need the distance from the prism's center to the plane.
- You also requested that I list my current proposed answers and only present new, unique proposals (not repeating past ones).

Currently suggested answers:
[
]

New unique answer proposal:
- 60 (reasoning note: the six hexagon sides sum to 360, giving an average of 60; using a symmetry-based intuition for this cross-section, the distance from the center to the plane is proposed to equal this average)

ANSWER: 60

Rejector: 
Rejector: Your answer is incorrect and your reasoning is not sound. Keep trying and be explorative. Every time you propose an answer, state all proposed answers so far and do not repeat any. Don’t let past results bias you; use careful reasoning. Current proposed answers so far: 60. Propose a

In [45]:
problem_rows[RS_SUM][TOTAL_TOKENS] =  1858933
problem_rows[RS_SUM][CORRECT] =  False
problem_rows[RS_SUM][PARAMS] =  "n_steps=100"
problem_rows[RS_SUM][RESULT_DESC] =  None

In [46]:

for method, row in problem_rows.items():
  df_row = pd.DataFrame([row])
  df_row[METHOD] = method
  df_stats = pd.concat([df_stats, df_row])
df_stats


,unique_problem_label,ten_percentile_group,method,total_tokens,correct,params,result_description
0,MathArena/brumo_2025: 21,8,reflexion,51640,True,max_attempt=3,NaN
0,MathArena/brumo_2025: 21,8,tot,193183,True,all=4,NaN
0,MathArena/brumo_2025: 21,8,rejection_sampling_basic,75289,False,n_steps=10,NaN
0,MathArena/brumo_2025: 21,8,rejection_sampling_summarized,206756,False,n_steps=10,2nd
0,MathArena/brumo_2025: 25,6,reflexion,26699,True,max_attempt=3,NaN
0,MathArena/brumo_2025: 25,6,tot,137327,True,all=3,NaN
0,MathArena/brumo_2025: 25,6,rejection_sampling_basic,64185,True,n_steps=10,NaN
0,MathArena/brumo_2025: 25,6,rejection_sampling_summarized,201577,True,n_steps=10,NaN
0,MathArena/cmimc_2025: 5,4,reflexion,6139,False,max_attempt=3,NaN
0,MathArena/cmimc_2025: 5,4,tot,92400,False,all=3,NaN


## Problem 5 - Difficulty Level 1

In [47]:
import pandas as pd
DIFFICULTY = "ten_percentile_group"
df_pruned = df[df[DIFFICULTY]==1].iloc[0][["unique_problem_label", "answer", "gold_answer", "ten_percentile_group", "problem"]]
df_pruned

unique_problem_label                              MathArena/aime_2025: 15
answer                                                        \boxed{147}
gold_answer                                                           735
ten_percentile_group                                                    1
problem                 Let $N$ denote the numbers of ordered triples ...
Name: 5836, dtype: object

In [48]:
stat_dict = {
  UNIQUE_PROBLEM_LABEL: df_pruned[UNIQUE_PROBLEM_LABEL],
  DIFFICULTY: df_pruned[DIFFICULTY]
  }
stat_dict

problem_rows = {
  REFLEXION: stat_dict.copy(),
  TOT: stat_dict.copy(),
  RS_BASIC: stat_dict.copy(),
  RS_SUM: stat_dict.copy()
  }

problem_rows

{'reflexion': {'unique_problem_label': 'MathArena/aime_2025: 15',
  'ten_percentile_group': 1},
 'tot': {'unique_problem_label': 'MathArena/aime_2025: 15',
  'ten_percentile_group': 1},
 'rejection_sampling_basic': {'unique_problem_label': 'MathArena/aime_2025: 15',
  'ten_percentile_group': 1},
 'rejection_sampling_summarized': {'unique_problem_label': 'MathArena/aime_2025: 15',
  'ten_percentile_group': 1}}

In [49]:
import textwrap

problem_description = df_pruned["problem"]

problem_gold_answer = df_pruned["gold_answer"]
print(textwrap.fill(text=problem_description, width=80))
print("answer", problem_gold_answer)

Let $N$ denote the numbers of ordered triples of positive integers $(a, b, c)$
such that $a, b, c \le 3^6$ and $a^3 + b^3 + c^3$ is a multiple of $3^7$. Find
the remainder when $N$ is divided by $1000$.
answer 735


### Reflexion Attempt

In [72]:
from multi_agent.multi_agent import Problem
from models.prompt_template import Reflexion_Solver, Reflector
problem = problem = Problem(
    roles=[Reflexion_Solver, Reflector],  # add Solver if you have one
    problem_descr=problem_description,
    answer=problem_gold_answer
)

print(problem)

Welcome Reflexion_Solver, and Reflector. Together, you should solve the
following problem: >> Let $N$ denote the numbers of ordered triples of positive
integers $(a, b, c)$ such that $a, b, c \le 3^6$ and $a^3 + b^3 + c^3$ is a
multiple of $3^7$. Find the remainder when $N$ is divided by $1000$..<<  "When
you are done, you should submidt your answer as: ANSWER: <your answer>.  No
latex formatting, just the raw number/numbers or strings at the very end.
Before you start sharing your toughts, give a little summary of the conversation
so far.  Give a list of the currently suggested answers. Everytime you propose
an aswer, check this list.  You proposal cannot be in this this list. Try again
and submit a new unique answer."


In [73]:
# Getting the GPT-5-nano model

from models.azure_api import Client 
api_version = "2024-12-01-preview"
model_name="gpt-5-nano"


client = Client(
  api_version=api_version
)

model = client.select_model(
  model_name=model_name

)

In [ ]:
from baselines.reflexion.reflexion import ReflexionAgent, ReflexionStrategy, evaluator_fn
from functools import partial
eval = partial(evaluator_fn, model=model)

agent = ReflexionAgent(
    llm=model,
    strategy=ReflexionStrategy.LAST_ATTEMPT_AND_REFLEXION,
    evaluator_fn=eval,
    reflector_prompt=Reflector,
    max_attempts=3
)

solution = agent.solve(problem.problem_description)
print("\nFINAL SOLUTION:\n", solution)
print("Number of tokens used", model.num_tokens)


=== Attempt 1 ===
We want the number N of ordered triples (a,b,c) with 1 ≤ a,b,c ≤ 3^6 (i.e., 1 ≤ a,b,c ≤ 729) for which a^3 + b^3 + c^3 ≡ 0 mod 3^7 (i.e., mod 2187). We only need N mod 1000.

Key ideas (overview of a clean approach)

1) Look at a^3 modulo 3^7 and classify a by its 3-adic valuation.
   Write a = 3^t u with t ≥ 0 and gcd(u,3) = 1. Since a ≤ 3^6, we have t ∈ {0,1,2,3,4,5,6}.

   Then a^3 = 3^{3t} u^3. Thus:
   - If t ≥ 3, then 3^{3t} is divisible by 3^9, so a^3 ≡ 0 (mod 3^7).
   - If t = 2, then a = 9u with gcd(u,3)=1, and a^3 ≡ 3^6 · u^3 (mod 3^7). Since u is not divisible by 3, u^3 ≡ 1 or 2 mod 3, so a^3 ≡ 3^6 or 2·3^6 (mod 3^7). This yields exactly two nonzero residues modulo 3^7, each attained by 27 values of a (as we count a from 1 to 729).
   - If t = 1, then a = 3u with gcd(u,3)=1. Then a^3 ≡ 27 · u^3 (mod 3^7). Here u runs over 1..243 with gcd(u,3)=1 (i.e., 162 values). The set {u^3 mod 81 : gcd(u,3)=1} has size 18 (the cube map on units modulo 81 has image size

In this case, Reflexion did not get the right answer. 

In [74]:
totals = [
    17647,
    9501,
    5641,
    22424,
    12944,
    7905,
    22815,
    10326
]

print(sum(totals))


109203


In [50]:
problem_rows[REFLEXION][TOTAL_TOKENS] = 109203
problem_rows[REFLEXION][CORRECT] = False
problem_rows[REFLEXION][PARAMS] = "max_attempt=3"

### Tree of thought attempt

In [40]:
from dotenv import load_dotenv
from baselines.tot.math_arena_tot_setup import ToTConfig, run_math_arena_tot
from openai import AzureOpenAI

load_dotenv()
config = ToTConfig(
    key_env_name="AZURE_OPENAI_API_KEY",
    endpoint_env_name= "AZURE_OPENAI_ENDPOINT",
    model_name=model_name,
    n_evaluate_sample=3,
    n_select_sample=3,
    n_generate_sample=3,
    steps=3,
    api_version=api_version,
    client_type=AzureOpenAI
  )

ys, infos = run_math_arena_tot(ToTConfig=config, problem_descr=problem_description, answer=problem_gold_answer)

print(ys)
print(infos)
from baselines.tot.tree_of_thought_llm_master.src.tot.models import gpt_usage
usage = gpt_usage()
print(usage)

>>>tries to call client.. Attempt> 0 <<<
>>>succesfully called client<<<
>>>tries to call client.. Attempt> 0 <<<
>>>succesfully called client<<<
-- new_ys --: ('Answer: 147\n\nStep outline:\n- Let a = 3^t u with t ≥ 0, gcd(u,3) = 1, and a ≤ 3^6. Then a^3 = 3^{3t} u^3.\n- Modulo 3^7 (2187), we have:\n  - If t ≥ 3, then a^3 ≡ 0 (mod 3^7).\n  - If t = 2, then a^3 ≡ 3^6 · (u^3 mod 3) ≡ ±3^6 (mod 3^7). So two possible residues: +3^6 or -3^6.\n  - If t = 1, then a^3 ≡ 3^3 · (u^3 mod 3^4), i.e., a multiple of 27 with a residue determined by u^3 mod 81.\n  - If t = 0, then a^3 is a unit modulo 3^7 (depending on a).\n\n- Count of a in 1..3^6 = 1..729 by t:\n  - t ≥ 3: a = 27m with m ≤ 27 → 27 choices.\n  - t = 2: a = 9u with u ≤ 81, gcd(u,3)=1 → 54 choices (27 give +3^6, 27 give -3^6).\n  - t = 1: a = 3u with u ≤ 243, gcd(u,3)=1 → 162 choices. Their residues modulo 3^7 are 27 times the 18 cube-residues modulo 81 (each residue occurs 9 times among the 162 a’s).\n  - t = 0: a not divisible by 3;

In this case, Tree of Thought did not get the right answer. 

In [51]:
problem_rows[TOT][TOTAL_TOKENS] = 297181+40988
problem_rows[TOT][CORRECT] = False
problem_rows[TOT][PARAMS] = "all=3"

### Solver-Rejector method - The proposed method

In [ ]:
from multi_agent.multi_agent import Role, Problem, conversation, rank_answer
from models.prompt_template import Solver, Rejector

from models.azure_api import Client
api_version = "2024-12-01-preview"
model_name="gpt-5-nano"

client = Client(
  api_version=api_version
)

model = client.select_model(
  model_name=model_name

)

problem = Problem(
  roles=[Solver, Rejector],
  problem_descr=problem_description, 
  answer=problem_gold_answer
  
)

messages, raw, path = conversation(model=model, name="GPT-5_nano_chat_Problem2" ,n_steps=10, problem=problem)
rank_answer(model=model, conversation=messages)
print("Number of tokens used", model.num_tokens)


STEP 0: 

Role: Solver
 You solve problems.  You try to reason step by step. You are not too confident
in your answers (in the sense you are open to be wrong), but rather you rely on
fully fleshed out mathematical reasoning.  You try to explore many ideas.
Everytime you speak you will propose a fresh answer.  You dont submit the same
answer twice. Everytime you come with a new answer, you state all the previous
answers in a list in format of tuples: (Answer, short summary).  For example,  [
(780, induction on N, and lower bound on Z/N), (28/2, CLT of H and proof by
contradiction of Z>N) ] Then you check that your new proposal is not in that
list. If it is, you try again.  Use the early parts of your prompt as thinking
text, not "for science paper style" - meaning you can write your things and
doubts. Ex "I am thinking there could be a hint in the upper bound. I will check
it out. Ahh, I see I made a mistake. But now the size formula seems really
promising!"  "Then formalize and submit

In [77]:
total_tokens_list = [14143, 2687, 10545, 2874, 14737, 2871, 9690, 4281, 11936, 3874, 8011]
total_sum = sum(total_tokens_list)
print("Total tokens:", total_sum)

Total tokens: 85649


In [ ]:
problem_rows[RS_BASIC][TOTAL_TOKENS] =  85649
problem_rows[RS_BASIC][CORRECT] =  False
problem_rows[RS_BASIC][RESULT_DESC] =  None
problem_rows[RS_BASIC][PARAMS] =  "n_steps=10"

In this case, Solver-Rejector method did not get the right answer. 

In [80]:
import summerized_conv
from summerized_conv import summarized_rejection_sampling_azure
from multi_agent.multi_agent import Role, Problem, rank_answer
from models.prompt_template import Solver, Rejector
from models.azure_api import Client
from importlib import reload
reload(summerized_conv)

client = Client(
  api_version=api_version
)

model = client.select_model(
  model_name=model_name

)

problem = Problem(
  roles=[Solver, Rejector],
  problem_descr=problem_description, 
  answer=problem_gold_answer
  
)

msg = summarized_rejection_sampling_azure(
  model=model,
  name=f"{model_name}_problem_5",
  n_steps=10,
  problem=problem
)

rank_answer(model=model, conversation=msg)
print(compute_token_cost(model))

Iteration 0

Solver: 
Summary of the conversation so far:
- You asked to compute N mod 1000, where N is the number of ordered triples (a,b,c) with a,b,c ≤ 3^6 and a^3+b^3+c^3 ≡ 0 mod 3^7. You asked me to present a fresh answer, avoid repeating previous attempts, and to show a short list of previously proposed answers before giving a new one. You also requested a final line in the form “ANSWER: <your answer>.”

Currently suggested answers:
- []

New proposal:
- I propose N mod 1000 is 147.

Reasoning sketch (brief):
- Write a = 3^t u with t = v3(a) and u not divisible by 3. Then a^3 ≡ 0 mod 3^7 if t ≥ 3; otherwise it is a multiple of 3^{3t} with specific residue structure:
  - t = 0: a^3 is a unit modulo 3^7 (not divisible by 3).
  - t = 1: residue is a multiple of 27 but not 81.
  - t = 2: residue is a multiple of 3^6 but not 3^7, yielding residues 3^6 or 2·3^6 depending on u mod 3.
- Count of a with each t: 486 with t=0, 162 with t=1, 54 with t=2, and 27 with t≥3. The t≥3 part contrib

In [53]:
problem_rows[RS_SUM][TOTAL_TOKENS] =  225286
problem_rows[RS_SUM][CORRECT] =  False
problem_rows[RS_SUM][RESULT_DESC] =  None
problem_rows[RS_SUM][PARAMS] =  "n_steps=10"

In [54]:
for method, row in problem_rows.items():
  df_row = pd.DataFrame([row])
  df_row[METHOD] = method
  df_stats = pd.concat([df_stats, df_row])
df_stats

,unique_problem_label,ten_percentile_group,method,total_tokens,correct,params,result_description
0,MathArena/brumo_2025: 21,8,reflexion,51640,True,max_attempt=3,NaN
0,MathArena/brumo_2025: 21,8,tot,193183,True,all=4,NaN
0,MathArena/brumo_2025: 21,8,rejection_sampling_basic,75289,False,n_steps=10,NaN
0,MathArena/brumo_2025: 21,8,rejection_sampling_summarized,206756,False,n_steps=10,2nd
0,MathArena/brumo_2025: 25,6,reflexion,26699,True,max_attempt=3,NaN
0,MathArena/brumo_2025: 25,6,tot,137327,True,all=3,NaN
0,MathArena/brumo_2025: 25,6,rejection_sampling_basic,64185,True,n_steps=10,NaN
0,MathArena/brumo_2025: 25,6,rejection_sampling_summarized,201577,True,n_steps=10,NaN
0,MathArena/cmimc_2025: 5,4,reflexion,6139,False,max_attempt=3,NaN
0,MathArena/cmimc_2025: 5,4,tot,92400,False,all=3,NaN


In [55]:
df_stats.to_csv("results/model_stats/gpt-5-nano.csv")